In [1]:
import os
import numpy as np
import matplotlib.pyplot as plt
import pandas as pd
from IPython.display import Audio, display
from pynwb import NWBHDF5IO
from collections import Counter

import ipywidgets as widgets
from ipywidgets import interact, interactive, fixed, IntSlider, FloatSlider, Dropdown, Checkbox

from brain_audio_decoder import BrainAudioDecoder
from custom_decoder import CustomBrainAudioDecoder
from brain_audio_decoder_viz import BrainAudioDecoderViz
from acoustic_change_detector import AcousticChangeDetector
from phoneme_validator import PhonemeValidator
from phonetic_dictionary import PhoneticDictionary
from hybrid_phoneme_models import HybridPhonemeModels
from unified_phoneme_pipeline import UnifiedPhonemePipeline
from markov_phoneme_model import MarkovPhonemeModel
from markov_children import  GMMHMMModel, HSMMModel
from simple_phoneme_models import SimplePhonemeModels

In [2]:
# Define paths
path_bids = './SingleWordProductionDutch-iBIDS'
path_output = './features'
path_results = './results'

# Create directories if they don't exist
os.makedirs(path_output, exist_ok=True)
os.makedirs(path_results, exist_ok=True)

In [3]:
use_augmentation=True
feature_extraction_method= 'high_gamma'  # 'multi_band'

# Try to load existing pipeline, otherwise create new one
try:
    # Try loading existing pipeline
    pipeline_hg = UnifiedPhonemePipeline.load_saved(path_results, method=feature_extraction_method)
    print(f"Loaded existing {feature_extraction_method} pipeline")
    
except (FileNotFoundError, AttributeError, TypeError) as e:
    # No existing pipeline found, create new one
    print(f"No existing {feature_extraction_method} pipeline found. Creating new one...")
    
    pipeline_hg = UnifiedPhonemePipeline(
        path_bids=path_bids,
        path_output=path_output,
        path_results=path_results,
        feature_extraction_method=feature_extraction_method,
        unknown_keep_ratio=0.1,
        debug_mode=False
    )
    
    # Run all steps
    print("Running pipeline steps...")
    pipeline_hg.step1_initialize_decoder()
    pipeline_hg.step2_stratify_participants()    
    pipeline_hg.step3_create_split()
    pipeline_hg.step4_initialize_detector()    
    pipeline_hg.step5_accumulate_data()
    pipeline_hg.step6_resolve_unknowns()
    pipeline_hg.step7_filter_unknowns()
    
    # Save the pipeline
    pipeline_hg.save()
    print(f"Created and saved new {feature_extraction_method} pipeline")

UnifiedPhonemePipeline: Initialized with DEBUG_MODE=False
Pipeline initialized
Loaded high_gamma pipeline
Loaded pipeline_high_gamma.pkl
Loaded existing high_gamma pipeline


In [9]:
use_augmentation=True
feature_extraction_method= 'multi_band'  # 'multi_band'

# Try to load existing pipeline, otherwise create new one
try:
    # Try loading existing pipeline
    pipeline_mb = UnifiedPhonemePipeline.load_saved(path_results, method=feature_extraction_method)
    print(f"Loaded existing {feature_extraction_method} pipeline")
    
except (FileNotFoundError, AttributeError, TypeError) as e:
    # No existing pipeline found, create new one
    print(f"No existing {feature_extraction_method} pipeline found. Creating new one...")
    
    pipeline_mb = UnifiedPhonemePipeline(
        path_bids=path_bids,
        path_output=path_output,
        path_results=path_results,
        feature_extraction_method=feature_extraction_method,
        unknown_keep_ratio=0.1,
        debug_mode=False
    )
    
    # Run all steps
    print("Running pipeline steps...")
    pipeline_mb.step1_initialize_decoder()
    pipeline_mb.step2_stratify_participants()    
    pipeline_mb.step3_create_split()
    pipeline_mb.step4_initialize_detector()    
    pipeline_mb.step5_accumulate_data()
    pipeline_mb.step6_resolve_unknowns()
    pipeline_mb.step7_filter_unknowns()
    
    # Save the pipeline
    pipeline_mb.save()
    print(f"Created and saved new {feature_extraction_method} pipeline")

UnifiedPhonemePipeline: Initialized with DEBUG_MODE=False
Pipeline initialized
Loaded multi_band pipeline
Loaded pipeline_multi_band.pkl
Loaded existing multi_band pipeline


In [7]:
# Get data
hg_train_data = pipeline_hg.get_training_data()
hg_test_data = pipeline_hg.get_test_data()

# Initialize simple models
simple_models = SimplePhonemeModels(debug_mode=True)

# Train all classical models
results = simple_models.train_all_classical(
    hg_train_data['features'],
    hg_train_data['phoneme_labels']
)

# Test each model
for model_name in simple_models.models:
    predictions, probs = simple_models.predict(hg_test_data['features'], model_name)
    accuracy = np.mean(predictions == hg_test_data['phoneme_labels'])
    print(f"{model_name} test accuracy: {accuracy:.4f}")

SimplePhonemeModels: Initialized with DEBUG_MODE=True
SimplePhonemeModels: Training naive_bayes...
SimplePhonemeModels: naive_bayes training accuracy: 0.3731
SimplePhonemeModels: Training logistic...
SimplePhonemeModels: logistic training accuracy: 0.7761
SimplePhonemeModels: Training lda...
SimplePhonemeModels: lda training accuracy: 0.7761
SimplePhonemeModels: Training knn...
SimplePhonemeModels: knn training accuracy: 0.2761
SimplePhonemeModels: Training linear_svm...


C:\Users\irina\anaconda3\Lib\site-packages\sklearn\svm\_base.py:1244: ConvergenceWarning: Liblinear failed to converge, increase the number of iterations.
  warnings.warn(


SimplePhonemeModels: linear_svm training accuracy: 0.7761
SimplePhonemeModels: Training random_forest...
SimplePhonemeModels: random_forest training accuracy: 0.7761
SimplePhonemeModels: Training gradient_boosting...
SimplePhonemeModels: gradient_boosting training accuracy: 0.7761
SimplePhonemeModels: Training lightgbm...
SimplePhonemeModels: lightgbm training accuracy: 0.7761
naive_bayes test accuracy: 0.0208
logistic test accuracy: 0.0312
lda test accuracy: 0.0417
knn test accuracy: 0.0521
linear_svm test accuracy: 0.0208
random_forest test accuracy: 0.0521
gradient_boosting test accuracy: 0.0521
lightgbm test accuracy: 0.0208


In [10]:
# Get data
mb_train_data = pipeline_mb.get_training_data()
mb_test_data = pipeline_mb.get_test_data()

# Initialize simple models
simple_models = SimplePhonemeModels(debug_mode=True)

# Train all classical models
results = simple_models.train_all_classical(
    mb_train_data['features'],
    mb_train_data['phoneme_labels']
)

# Test each model
for model_name in simple_models.models:
    predictions, probs = simple_models.predict(mb_test_data['features'], model_name)
    accuracy = np.mean(predictions == mb_test_data['phoneme_labels'])
    print(f"{model_name} test accuracy: {accuracy:.4f}")

SimplePhonemeModels: Initialized with DEBUG_MODE=True
SimplePhonemeModels: Training naive_bayes...
SimplePhonemeModels: naive_bayes training accuracy: 0.5899
SimplePhonemeModels: Training logistic...
SimplePhonemeModels: logistic training accuracy: 0.9784
SimplePhonemeModels: Training lda...
SimplePhonemeModels: lda training accuracy: 0.9424
SimplePhonemeModels: Training knn...
SimplePhonemeModels: knn training accuracy: 0.2590
SimplePhonemeModels: Training linear_svm...


C:\Users\irina\anaconda3\Lib\site-packages\sklearn\svm\_base.py:1244: ConvergenceWarning: Liblinear failed to converge, increase the number of iterations.
  warnings.warn(


SimplePhonemeModels: linear_svm training accuracy: 0.9784
SimplePhonemeModels: Training random_forest...
SimplePhonemeModels: random_forest training accuracy: 0.9784
SimplePhonemeModels: Training gradient_boosting...
SimplePhonemeModels: gradient_boosting training accuracy: 0.9784
SimplePhonemeModels: Training lightgbm...
SimplePhonemeModels: lightgbm training accuracy: 0.9784
naive_bayes test accuracy: 0.0208
logistic test accuracy: 0.0312
lda test accuracy: 0.0208
knn test accuracy: 0.0312
linear_svm test accuracy: 0.0000
random_forest test accuracy: 0.0312
gradient_boosting test accuracy: 0.0312
lightgbm test accuracy: 0.0208
